<a href="https://colab.research.google.com/github/kacp-i/BCU-Work/blob/main/Y2%20S2%20CMP5366%20-%20Data%20Management%20and%20Machine%20Learning%20Operations/Pre_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library Imports

In [ ]:
# importing libraries used throughout the training & model
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style('whitegrid')

from scipy.stats import skewnorm

# libraries used for encoding data for the dataset, training different models and evaluating each model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from statsmodels.graphics.mosaicplot import mosaic

import warnings
warnings.filterwarnings('ignore')

# Importing the dataset

In [ ]:
# mounting the google drive - can comment this out and use the upload feature below
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# grab the csv file from the google drive
ds = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/S2 Assessment/S4 Dataset.csv', encoding='cp1252')

In [ ]:
# alternative upload method to google drive
# from google.colab import files
# file = files.upload()

#Exploring the dataset

In [ ]:
# shows the first 5 rows of the dataset & the column headers
ds.head()

In [ ]:
# shows all of the columns and their datatype and how many non-null cells they have
ds.info()

**Describing the Columnns:**

- Zone: Differnet zones of the world
- Country: Country name
- ISO3: 3 letter country codes
- Year: The year of the data collected
- Population: Population of the country at the time (rounded)
- Fortification: The amount of micronutrient fortified
- PCDEA: Per capita daily energy availability
- MDI: Micronutrient Density Index
- Tagname: Abbreviation of the micronutrient name
- Micronutrient: Name of micronutrient
- Units: Units of measurement of micronutrient
- Estimated Intake: The estimated intake for that micronutrient
- Requirements: The required amount of micronutrient
- Prevalence of Inadequate Intake: Amount of micronutrient missing

# Abstracting the data

In [ ]:
# showing the dataset information
ds.info()

In [ ]:
# dropping useless columns
ds = ds.drop(['Zone','ISO3','Fortification','Tagname'], axis=1)

# these columns are being dropped due to the lack of meaningful data inside of them

In [ ]:
# checking for null values across the dataset
ds.isnull().sum()

# Data Manipulation

In [ ]:
# describing the dataset pre-encoding (only shows numerical columns)
ds.describe()

In [ ]:
# checking which columns need to be converted from object datatype to int datatype
ds.info()

In this case its all the columns with a different datatype than int64 or float64

- Country
- Micronutrient
- Units

# Country Column Encoding

In [ ]:
# checking the values and their amounts in the column
ds['Country'].value_counts()

In [ ]:
# creating a variable for the encoder
lblMake = LabelEncoder()

# grab each unique variable from the column and turn it into its numerical version
lblMake.fit(ds['Country'].drop_duplicates())

# create a new column and apply the encoding values
ds['countryEncoded'] = lblMake.transform(ds['Country'])

In [ ]:
# checking which numerical value corresponds to the non-numerical value
ds[['Country', 'countryEncoded']].value_counts()

In [ ]:
# create a new file to hold
# as there are too many unique values to show as an output - you can reference the .txt file
# when using user predictions
makeFile = open("country.txt", "w")

# write the non-numerical and numerical values to a text file
makeFile.write(ds[['Country', 'countryEncoded']].to_string())

# close the file to save it
makeFile.close()

In [ ]:
# dropping the old column from the dataset as it is unnecessary
ds = ds.drop('Country', axis=1)

# Micronutrient Column Encoding

In [ ]:
# checking the values and their amounts in the column
ds['Micronutrient'].value_counts()

In [ ]:
# creating a variable for the encoder
lblMake = LabelEncoder()

# grab each unique variable from the column and turn it into its numerical version
lblMake.fit(ds['Micronutrient'].drop_duplicates())

# create a new column and apply the encoding values
ds['micronutrientEncoded'] = lblMake.transform(ds['Micronutrient'])

In [ ]:
# checking which numerical value corresponds to the non-numerical value
ds[['Micronutrient', 'micronutrientEncoded']].value_counts()

In [ ]:
# dropping the old column from the dataset as it is unnecessary
ds = ds.drop('Micronutrient', axis=1)

# Units Column Encoding

In [ ]:
# checking the values and their amounts in the column
ds['Units'].value_counts()

In [ ]:
# creating a variable for the encoder
lblMake = LabelEncoder()

# grab each unique variable from the column and turn it into its numerical version
lblMake.fit(ds['Units'].drop_duplicates())

# create a new column and apply the encoding values
ds['unitsEncoded'] = lblMake.transform(ds['Units'])

In [ ]:
# checking which numerical value corresponds to the non-numerical value
ds[['Units', 'unitsEncoded']].value_counts()

In [ ]:
# dropping the old column from the dataset as it is unnecessary
ds = ds.drop('Units', axis=1)

# Final Check

In [ ]:
# checking each columns datatype
ds.info()

In [ ]:
# describing the dataset
ds.describe()

# Cleaning Up The Dataset

In [ ]:
# renaming columns to make it more meaningful
ds.rename(columns = {'Year' : 'year',
                     'Population' : 'popu',
                     'Estimated Intake' : 'est intake',
                     'Requirements' : 'req',
                     'Prevalence of Inadequate Intake' : 'prev inadeq intake',
                     'countryEncoded' : 'cntry',
                     'micronutrientEncoded' : 'micro',
                     'unitsEncoded' : 'units'},
          inplace = True)

In [ ]:
# checking if changes have been applied
ds.info()

# Visualisation & Clearing Outliers

In [ ]:
# describing the dataset
ds.describe()

In [ ]:
# checking for outliers in the population column
# as there is a huge difference between the mean and max value
sns.boxplot(ds['popu'])
plt.title('Original Box Plot of Population')
plt.show()

In [ ]:
# adding a threshold to values of the column
removed_outliers = ds[ds['popu'] <= 100000000]

# creating a new boxplot for the column with the threshold applied
sns.boxplot(removed_outliers['popu'])
plt.title('Box Plot with limited outliers of Population')
plt.show()

# Training AI Models

In [ ]:
ds.info()

In [ ]:
# selecting the columns to use for training the model
dsLearn = ds[['year',
              'popu',
              'PCDEA',
              'MDI',
              'req',
              'cntry',
              'micro',
              'est intake']]

In [ ]:
# allocating the columns from  to  as x - the data used to predict
x = dsLearn.iloc[:, 0 : -1]

x.head() # checking if the correct columns have been selected

In [ ]:
# allocating the column Value as y - what i am trying to predict
y = dsLearn.iloc[:, -1:, ]

y.head() # checking if the correct columns have been selected

In [ ]:
# separating the data into test and train data
xTrain, xTest, yTrain, yTest = train_test_split(x, y, random_state=42, test_size=0.2)

# Linear Regression

In [ ]:
# initalise and train the linear regression model
lrModel = LinearRegression()
lrModel.fit(xTrain, yTrain)

# make predictions
yPrediction = lrModel.predict(xTest)

# evaluating the model
mse = mean_squared_error(yPrediction, yTest)
sqrt_mse = np.sqrt(mse)
mae = mean_absolute_error(yPrediction, yTest)
r2 = r2_score(yTest, yPrediction)
coefficient = lrModel.score(xTrain, yTrain)
print(f"MSE : {mse:.3f} \nMSE_SQRT : {sqrt_mse:.3f} \nMAE : {mae:.3f} \nR2 Score : {r2:.3f} \nCoefficient : {coefficient:.3f}")

In [ ]:
# showing statistics of the model
print("Intercept:", lrModel.intercept_,
      "\nCoeficcient:", lrModel.coef_,
      "\nScore of the training data:", lrModel.score(xTrain, yTrain),
      "\nScore of the test data:", lrModel.score(xTest, yTest))

# Decision Tree Regression

In [ ]:
# initialise and train the Decision Tree Regressor model
dtModel = DecisionTreeRegressor(random_state=42)
dtModel.fit(xTrain, yTrain)

# make predictions
y_pred_dt = dtModel.predict(xTest)

# evaluating the model
mse_dt = mean_squared_error(yTest, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
mae = mean_absolute_error(y_pred_dt, yTest)
r2_dt = r2_score(yTest, y_pred_dt)

print("Decision Tree Regressor Evaluation Metrics:")
print(f"Mean Squared Error (MSE): {mse_dt:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_dt:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2 Score): {r2_dt:.2f}") # shows how much of an impact the independant variables (features) have on the dependant value (prediction)

In [ ]:
# showing statistics of the model
print("Score of the train data:",
      dtModel.score(xTrain, yTrain),
      "\nScore of the test data:",
      dtModel.score(xTest, yTest))

# Random Forest Regression

In [ ]:
# initialise and train the Random Forest Regressor model
rfModel = RandomForestRegressor(random_state=42, n_estimators=100)
rfModel.fit(xTrain, yTrain)

# make predictions
y_pred_rf = rfModel.predict(xTest)

# evaluating the model
mse_rf = mean_squared_error(yTest, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae = mean_absolute_error(y_pred_rf, yTest)
r2_rf = r2_score(yTest, y_pred_rf)

print("Random Forest Regressor Evaluation Metrics:")
print(f"Mean Squared Error (MSE): {mse_rf:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2 Score): {r2_rf:.2f}")

In [ ]:
# showing statistics of the model
print("Score of the train data:", rfModel.score(xTrain, yTrain),
      "\nScore of the test data:", rfModel.score(xTest, yTest))

# Lasso Regression

In [ ]:
# initialising and fitting the training data onto the model
laModel = Lasso(alpha= 0.4)
laModel.fit(xTrain, yTrain)

# make predictions
yPred = laModel.predict(xTest)

# evaluating the model
mse = round(mean_squared_error(yPred, yTest), 2 )
rmse = np.sqrt(mse)
mae = mean_absolute_error(yPred, yTest)
r2 = r2_score(yPred, yTest)

print('Lasso Regression Evaluation Metrics')
print('Mean Squared Error : ', mse)
print('Root Mean Squared Error',  rmse)
print('Mean Absolute Error : ', mae)
print('R2_Squared  ', r2)

In [ ]:
# showing statistics of the model
print("Score of the train data:", laModel.score(xTrain, yTrain),
      "\nScore of the test data:", laModel.score(xTest, yTest))

# Predictions Using User Input

**The values needed for predicting a value:**
- Year
- Population
- PCDEA
- MDI
- Required Intake
- Country
- Micronutrient

**Values for Country are available in the country.txt file**

**Values for Micronutrient:**
- Calcium	= 0
- Copper = 1
- Folate = 2
- Iron = 3
- Magnesium =	4
- Niacin = 5
- Phosphorus = 6
- Riboflavin = 7
- Thiamin =	8
- Vitamin A =	9
- Vitamin B12 =	10
- Vitamin B6 =	11
- Vitamin C =	12
- Zinc =	13

In [ ]:
# using linear regression to predict the value of a motorbike based on given parameters

val1 = lrModel.predict([
    [2000, 10000000, 2997, 0.7, 0.5, 6, 3] # change variables here to have different values on output
    ])
value1 = val1[0][0]

print(f'Predicted Prevalence of Inadequate Intake = {value1:.2f}')

In [ ]:
# using decision tree regression to predict the value of a motorbike based on given parameters

val2 = dtModel.predict([
    [2500, 125, 111, 1, 1, 1, 2019] # change variables here to have different values on output
    ])
value2 = val2[0]

print(f'Predicted Prevalence of Inadequate Intake = {value2:.2f}')

In [ ]:
# using random forest regression to predict the value of a motorbike based on given parameters

val3 = rfModel.predict([
    [2500, 125, 111, 1, 1, 1, 2019] # change variables here to have different values on output
    ])
value3 = val3[0]

print(f'Predicted Prevalence of Inadequate Intake = {value3:.2f}')

In [ ]:
# using lasso regression to predict the value of a motorbike based on given parameters

val4 = laModel.predict([
    [2500, 125, 111, 1, 1, 1, 2019] # change variables here to have different values on output
    ])
value4 = val4[0]

print(f'Predicted Prevalence of Inadequate Intake = {value4:.2f}')